# Vígil.ia — Auto-treino com vídeos brutos (v4)

**Ideia:** o RT-DETR ft_v3 anota os teus vídeos brutos sozinho, usando o
**veredito travado** como controle de qualidade — a classe de cada grão vem do
consenso de dezenas de frames (ponderado pela confiança), não de um chute isolado.
Só entram no dataset tracks confiáveis; frame com grão duvidoso é descartado.

Isso ensina o **domínio real de vídeo** (borrão, luz, distância) e reforça o
`spotted`: o grão manchado é rotulado pelo consenso (frames nítidos pesam mais)
e treinado também nos frames borrados.

**Preparação (você):** gravar vídeos variados e subir para `MyDrive/videos_soja/`
- lotes bons, ruins e misturados · distâncias e luzes diferentes · movimento lento E médio
- 20–40 s cada; quanto mais variedade, melhor
- **um deles fica de fora do treino** (holdout) e vira a prova final A/B

Pré-requisitos no Drive: `soja_rtdetr_ft_v3.pt`, fotos reais (dataset v3), pasta `videos_soja/`.

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Caminhos e parâmetros

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')

VIDEOS_DIR = '/content/drive/MyDrive/videos_soja'
videos = sorted(glob.glob(f'{VIDEOS_DIR}/*.mp4') + glob.glob(f'{VIDEOS_DIR}/*.MP4')
                + glob.glob(f'{VIDEOS_DIR}/*.mov') + glob.glob(f'{VIDEOS_DIR}/*.MOV'))
assert len(videos) >= 2, f'Preciso de pelo menos 2 vídeos em {VIDEOS_DIR} (1 vira holdout). Achei: {videos}'

# Escolha EXPLÍCITA do vídeo de prova (holdout) — deixe vazio ('') para cair no
# padrão (último em ordem alfabética), mas prefira nomear na mão: assim você
# sempre sabe, com certeza, qual vídeo o modelo NUNCA viu no treino.
HOLDOUT_NAME = ''   # ex.: 'lote_misto_03.mp4'

if HOLDOUT_NAME:
    matches = [v for v in videos if os.path.basename(v) == HOLDOUT_NAME]
    assert matches, f'HOLDOUT_NAME={HOLDOUT_NAME!r} não achado em {VIDEOS_DIR}. Arquivos: {[os.path.basename(v) for v in videos]}'
    HOLDOUT = matches[0]
else:
    HOLDOUT = videos[-1]  # padrão: último em ordem alfabética
TRAIN_VIDEOS = [v for v in videos if v != HOLDOUT]

print('=' * 60)
print('TREINO  :', [os.path.basename(v) for v in TRAIN_VIDEOS])
print('HOLDOUT :', os.path.basename(HOLDOUT), '<- este NUNCA entra no treino')
print('=' * 60)

RTDETR_PT = '/content/drive/MyDrive/soja_rtdetr_ft_v3.pt'
assert os.path.exists(RTDETR_PT), 'soja_rtdetr_ft_v3.pt não encontrado no Drive!'

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

# Controle de qualidade do auto-rótulo
TRACK_MIN_FRAMES = 10   # track precisa existir em >=10 frames
TRACK_LOCK_RATIO = 0.60 # e ter >=60% de consenso de classe
FRAME_STRIDE = 5        # grava 1 frame a cada 5 (evita quase-duplicatas)
CONF_TRACK = 0.35

## 3. Modelo professor + patch NMS

In [ ]:
import numpy as np
import torchvision
from ultralytics import RTDETR
from ultralytics.models.rtdetr.predict import RTDETRPredictor

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']

if not hasattr(RTDETRPredictor, '_pp_orig'):
    RTDETRPredictor._pp_orig = RTDETRPredictor.postprocess

def _pp_nms(self, preds, img, orig_imgs):
    results = RTDETRPredictor._pp_orig(self, preds, img, orig_imgs)
    for r in results:
        if len(r.boxes) > 1:
            keep = torchvision.ops.nms(r.boxes.xyxy, r.boxes.conf, iou_threshold=0.6)
            r.update(boxes=r.boxes.data[keep])
    return results

RTDETRPredictor.postprocess = _pp_nms
teacher = RTDETR(RTDETR_PT)
print('professor pronto ✅')

## 4. Auto-rotulador: vídeo → frames rotulados por consenso
Duas passadas por vídeo: (1) rastreia tudo e fecha o veredito de cada track;
(2) relê o vídeo e grava os frames amostrados com as caixas — classe = veredito
do track. Frame que contém grão de track NÃO-confiável é descartado inteiro
(melhor perder frame do que ensinar erro).

In [ ]:
import cv2
from collections import defaultdict, Counter

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def autolabel_video(video_path, out_dir, tag):
    os.makedirs(f'{out_dir}/images/train', exist_ok=True)
    os.makedirs(f'{out_dir}/labels/train', exist_ok=True)

    # ---- passada 1: rastreia e acumula votos por track ----
    votes = defaultdict(Counter)
    per_frame = {}  # idx -> [(tid, xyxy, conf)]
    idx = 0
    for r in teacher.track(source=video_path, conf=CONF_TRACK, tracker='bytetrack.yaml',
                           stream=True, persist=True, verbose=False):
        dets = []
        if r.boxes.id is not None:
            for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                                        r.boxes.id.int().tolist(),
                                        r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist()):
                votes[tid][c] += cf
                dets.append((tid, xyxy, cf))
        per_frame[idx] = dets
        idx += 1

    # veredito por track + filtro de qualidade
    frames_per_track = Counter()
    for dets in per_frame.values():
        for tid, _, _ in dets:
            frames_per_track[tid] += 1
    verdict = {}
    for tid, cnt in votes.items():
        c, n = cnt.most_common(1)[0]
        if frames_per_track[tid] >= TRACK_MIN_FRAMES and n >= TRACK_LOCK_RATIO * sum(cnt.values()):
            verdict[tid] = c
    print(f'  {tag}: {len(votes)} tracks, {len(verdict)} confiáveis')

    # ---- passada 2: relê o vídeo e grava frames amostrados ----
    cap = cv2.VideoCapture(video_path)
    written = dropped = 0
    cls_count = Counter()
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        dets = per_frame.get(idx, [])
        idx += 1
        if (idx - 1) % FRAME_STRIDE or not dets:
            continue
        if any(tid not in verdict for tid, _, _ in dets):
            dropped += 1  # frame com grão duvidoso -> fora
            continue
        h0, w0 = frame.shape[:2]
        lb, s, left, top = letterbox640(frame)
        lines = []
        for tid, (x1, y1, x2, y2), _ in dets:
            cx = ((x1 + x2) / 2 * s + left) / 640.0
            cy = ((y1 + y2) / 2 * s + top) / 640.0
            ww = (x2 - x1) * s / 640.0
            hh = (y2 - y1) * s / 640.0
            c = verdict[tid]
            cls_count[NAMES[c]] += 1
            lines.append(f'{c} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        stem = f'{tag}_{idx:06d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write('\n'.join(lines))
        written += 1
    cap.release()
    print(f'  {tag}: {written} frames gravados, {dropped} descartados | caixas: {dict(cls_count)}')
    return cls_count

VID_DIR = '/content/soja_vid'
total = Counter()
for v in TRAIN_VIDEOS:
    tag = os.path.splitext(os.path.basename(v))[0].replace(' ', '_')
    total += autolabel_video(v, VID_DIR, tag)
print('\nTOTAL de caixas por classe (vídeo):', dict(total))
print('⚠️ Se spotted/immature estiverem baixos aqui, grave mais vídeos com esses grãos.')

## 5. Sanidade visual — frames auto-rotulados (OLHAR antes de treinar)

In [ ]:
import matplotlib.pyplot as plt

sample = sorted(glob.glob(f'{VID_DIR}/images/train/*.jpg'))
sample = sample[:: max(1, len(sample) // 6)][:6]
plt.figure(figsize=(13, 8))
for i, p in enumerate(sample):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for ln in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
        c, cx, cy, ww, hh = ln.split()
        cx, cy, ww, hh = float(cx), float(cy), float(ww), float(hh)
        x1, y1 = int((cx - ww / 2) * w), int((cy - hh / 2) * h)
        cv2.rectangle(img, (x1, y1), (int((cx + ww / 2) * w), int((cy + hh / 2) * h)), (0, 255, 0), 2)
        cv2.putText(img, NAMES[int(c)], (x1, max(14, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)
    ax = plt.subplot(2, 3, i + 1); ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Dataset v4 = fotos (v3) + frames de vídeo
O `train` do data.yaml aceita LISTA de pastas — junta os dois sem copiar nada.
Se o v3 não existir nesta sessão, reconstrói (células do notebook `melhoria_rtdetr_v3`).

In [ ]:
import yaml

if not os.path.exists('/content/soja_det_v3/data.yaml'):
    print('Dataset v3 não está nesta sessão — rode o notebook melhoria_rtdetr_v3.ipynb')
    print('(seções 3–4) OU cole aqui as células do builder v3 e execute build_v3().')
    raise SystemExit

V4_YAML = '/content/soja_v4.yaml'
yaml.safe_dump({
    'train': ['/content/soja_det_v3/images/train', f'{VID_DIR}/images/train'],
    'val': '/content/soja_det_v3/images/val',
    'names': {i: n for i, n in enumerate(NAMES)},
}, open(V4_YAML, 'w'), sort_keys=False, allow_unicode=True)
print('v4 pronto:', V4_YAML)

## 7. Fine-tune v4 (a partir do ft_v3, lr baixo)

In [ ]:
ft4 = RTDETR(RTDETR_PT)
ft4.train(
    data=V4_YAML, epochs=30, imgsz=640, batch=12, device=0, seed=42,
    optimizer='AdamW', lr0=3e-5, patience=12, close_mosaic=6,
    amp=False, warmup_epochs=0.0,   # receita anti-colapso validada
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_rtdetr', name='ft_v4', exist_ok=True,
)

!cp /content/runs/detect/runs_rtdetr/ft_v4/weights/best.pt /content/drive/MyDrive/soja_rtdetr_ft_v4.pt
print('backup ok: soja_rtdetr_ft_v4.pt')

## 8. Prova final — A/B no vídeo HOLDOUT (nunca visto no treino)
Gera `holdout_v3.mp4` (modelo antigo) e `holdout_v4.mp4` (novo) no Drive.
Assista os dois: o v4 deve segurar as classes melhor em movimento, com spotted incluso.

In [ ]:
LOCK_MIN_FRAMES = 8
LOCK_RATIO = 0.60
COLORS = {
    'intact': (80, 200, 80), 'immature': (60, 200, 200),
    'broken': (200, 100, 160), 'skin-damaged': (60, 160, 255),
    'spotted': (80, 80, 230),
}

def locked_video(model, src, out_path, conf=0.35):
    votes = defaultdict(Counter)
    frames_seen = Counter()
    locked = {}
    cap = cv2.VideoCapture(src)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    writer = None
    for r in model.track(source=src, conf=conf, tracker='bytetrack.yaml',
                         stream=True, persist=True, verbose=False):
        frame = r.orig_img.copy()
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        if r.boxes.id is not None:
            for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                                        r.boxes.id.int().tolist(),
                                        r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist()):
                x1, y1, x2, y2 = xyxy
                if tid not in locked:
                    votes[tid][NAMES[c]] += cf
                    frames_seen[tid] += 1
                    top, n = votes[tid].most_common(1)[0]
                    if frames_seen[tid] >= LOCK_MIN_FRAMES and n >= LOCK_RATIO * sum(votes[tid].values()):
                        locked[tid] = top
                if tid in locked:
                    cls = locked[tid]
                    color = COLORS[cls]
                    label = f'#{tid} {cls}'
                else:
                    color = (160, 160, 160)
                    label = f'#{tid} analisando…'
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(18, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        writer.write(frame)
    writer.release()
    for tid, cnt in votes.items():
        locked.setdefault(tid, cnt.most_common(1)[0][0])
    intact = sum(1 for c in locked.values() if c == 'intact')
    pct = intact / len(locked) if locked else 0
    print(f'  grãos: {len(locked)} | intactos: {intact} ({pct*100:.0f}%)'
          f' -> {"APROVADO" if pct >= 0.9 else "REPROVADO"}')
    return locked

novo = RTDETR('/content/runs/detect/runs_rtdetr/ft_v4/weights/best.pt')
print('ft_v3 (antigo) no holdout:')
locked_video(teacher, HOLDOUT, '/content/holdout_v3.mp4')
print('ft_v4 (novo) no holdout:')
locked_video(novo, HOLDOUT, '/content/holdout_v4.mp4')

!cp /content/holdout_v3.mp4 /content/holdout_v4.mp4 /content/drive/MyDrive/
print('\n2 vídeos no Drive: holdout_v3.mp4 (antes) e holdout_v4.mp4 (depois)')

## Como julgar e iterar

- **Melhorou?** Salva o v4 como campeão e repete o ciclo: mais vídeos brutos → v5.
  Cada rodada é barata (professor anota sozinho). Este é o motor de melhoria contínua.
- **Cuidado com eco:** o professor ensina o aluno com os próprios acertos — erros
  sistemáticos podem se reforçar. Por isso o holdout + teu olho continuam sendo o juiz.
  Se notar um erro repetido (ex.: spotted sumindo), a correção é dado NOVO daquele caso
  (gravar vídeos só de spotted), não mais auto-treino do mesmo material.
- Este mesmo dataset v4 serve depois pra **destilar o YOLO11s** (modelo local do i3).